# 04 Delay Analysis

Arrival and departure delays by train, station and date, on the cleaned layer from notebook 03.

> **Synthetic data.** All records are simulated and the findings are illustrative only. Trains and stations are shown by ID.

On time = delay of 5 minutes or less.

## 1. Setup

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from pathlib import Path

# project root is the folder that holds data/, whether the notebook runs from the root or from notebooks/
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = PROJECT_ROOT / "railpulse.duckdb"
RAW_TABLES = ["train_run_events", "train_master", "station_master"]
CLEAN_TABLES = [f"{t}_clean" for t in RAW_TABLES]
ON_TIME_THRESHOLD_MIN = 5

SURFACE, INK_SOFT, MUTED, AXIS = "#ffffff", "#52606d", "#9aa5b1", "#cbd2d9"
ARRIVAL, DEPARTURE = "#2f6db5", "#d9822b"
BLUE_RAMP = ["#eaf1fa", "#bcd3ee", "#8bb3e0", "#5a92d2", "#2f6db5", "#1d4a82"]


def use_theme():
    pio.templates["railpulse"] = go.layout.Template(layout=go.Layout(
        font=dict(family="Helvetica, Arial, sans-serif", size=13, color="#1f2933"),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, colorway=[ARRIVAL, DEPARTURE, MUTED],
        xaxis=dict(gridcolor="#eceff2", linecolor=AXIS, zeroline=False),
        yaxis=dict(gridcolor="#eceff2", linecolor=AXIS, zeroline=False),
        margin=dict(t=90, l=60, r=30, b=60),
    ))
    pio.templates.default = "railpulse"


def titled(fig, title, subtitle, height=420):
    fig.update_layout(title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>", x=0), height=height)
    return fig


def save(fig, name):
    """Charts are shown in the notebook only, nothing is written to disk."""
    return fig



def connect(read_only=True, require=()):
    """Open the project database and fail fast if a required table is missing."""
    assert DB_PATH.is_file(), f"{DB_PATH.name} not found, run 01_data_setup.ipynb first"
    con = duckdb.connect(str(DB_PATH), read_only=read_only)
    existing = {row[0] for row in con.sql("SELECT table_name FROM information_schema.tables").fetchall()}
    missing = [t for t in require if t not in existing]
    assert not missing, f"Missing tables {missing}, run the earlier notebooks first"
    return con

con = connect(read_only=True, require=CLEAN_TABLES)


def query(sql):
    return con.sql(sql).df()


pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
# shared plotly template
use_theme()

print(f"Connected to {DB_PATH.relative_to(PROJECT_ROOT)} (read-only)")
print(f"On-time threshold: {ON_TIME_THRESHOLD_MIN} minutes")

Connected to railpulse.duckdb (read-only)
On-time threshold: 5 minutes


## 2. Analysis base

In [2]:
con.execute(f"""
    -- long format, one row per stop per measure. everything below reads from this
    CREATE OR REPLACE TEMP VIEW delay_obs AS
    WITH events AS (
        -- a run is disrupted if any of its stops is flagged
        SELECT *, BOOL_OR(simulated_disruption_flag) OVER (PARTITION BY service_id) AS run_disrupted
        FROM train_run_events_clean
    )
    SELECT
        e.service_id, e.service_date, e.train_id, e.station_id, s.region, e.station_sequence,
        'arrival' AS measure,
        e.scheduled_arrival AS scheduled_time,
        e.arrival_delay_min AS delay_min,
        -- early counts as on time
        e.arrival_delay_min <= {ON_TIME_THRESHOLD_MIN} AS is_on_time,
        e.simulated_disruption_flag AS is_disrupted,
        e.run_disrupted,
        e.simulated_disruption_type AS disruption_type
    FROM events e
    JOIN station_master_clean s USING (station_id)
    WHERE e.arrival_delay_min IS NOT NULL
    UNION ALL
    SELECT
        e.service_id, e.service_date, e.train_id, e.station_id, s.region, e.station_sequence,
        'departure',
        e.scheduled_departure,
        e.departure_delay_min,
        e.departure_delay_min <= {ON_TIME_THRESHOLD_MIN},
        e.simulated_disruption_flag,
        e.run_disrupted,
        e.simulated_disruption_type
    FROM events e
    JOIN station_master_clean s USING (station_id)
    WHERE e.departure_delay_min IS NOT NULL
""")

# quick look at what's in the view
query("""
    SELECT
        measure,
        COUNT(*) AS observations,
        COUNT(DISTINCT service_id) AS runs,
        COUNT(DISTINCT train_id) AS trains,
        COUNT(DISTINCT station_id) AS stations,
        MIN(service_date) AS first_date,
        MAX(service_date) AS last_date
    FROM delay_obs
    GROUP BY measure
    ORDER BY measure
""")

,measure,observations,runs,trains,stations,first_date,last_date
0,arrival,2034,426,10,19,2026-08-01,2026-09-14
1,departure,2034,426,10,17,2026-08-01,2026-09-14


## 3. Headline measures

In [3]:
# network-wide numbers
kpi = query("""
    SELECT
        measure,
        COUNT(*) AS observations,
        ROUND(AVG(delay_min), 2) AS avg_delay_min,
        MEDIAN(delay_min) AS median_delay_min,
        -- p90 as well, the distribution has a long right tail
        quantile_cont(delay_min, 0.9) AS p90_delay_min,
        MAX(delay_min) AS max_delay_min,
        ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct,
        COUNT(*) FILTER (WHERE NOT is_on_time) AS late_observations
    FROM delay_obs
    GROUP BY measure
    ORDER BY measure
""")
kpi

,measure,observations,avg_delay_min,median_delay_min,p90_delay_min,max_delay_min,on_time_pct,late_observations
0,arrival,2034,2.70,2.0,8.0,33,79.7,413
1,departure,2034,4.05,4.0,10.0,36,68.6,638


In [4]:
distribution = query("""
    SELECT measure, delay_min, COUNT(*) AS observations
    FROM delay_obs
    GROUP BY measure, delay_min
    ORDER BY measure, delay_min
""")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.14,
                    subplot_titles=("Arrival delay", "Departure delay"))
for row, (measure, colour) in enumerate([("arrival", ARRIVAL), ("departure", DEPARTURE)], start=1):
    part = distribution[distribution["measure"] == measure]
    fig.add_trace(go.Bar(
        x=part["delay_min"], y=part["observations"], marker=dict(color=colour, cornerradius=3),
        hovertemplate="%{x} min: %{y} observations<extra></extra>", showlegend=False,
    ), row=row, col=1)
    # +0.5 so the line falls between the 5 and 6 minute bars
    fig.add_vline(x=ON_TIME_THRESHOLD_MIN + 0.5, line=dict(color=MUTED, width=1), row=row, col=1)
    fig.update_yaxes(title_text="Observations", row=row, col=1)
fig.add_annotation(x=ON_TIME_THRESHOLD_MIN + 1, y=1, xref="x", yref="y domain", xanchor="left", yanchor="top",
                   text=f"Late: more than {ON_TIME_THRESHOLD_MIN} min", showarrow=False, font=dict(color=INK_SOFT, size=12))
fig.update_xaxes(title_text="Delay (minutes)", row=2, col=1)
fig.update_layout(bargap=0.15)
fig.update_annotations(selector=dict(xref="x domain"), font=dict(color=INK_SOFT, size=13))
titled(fig, "Most stops run within a few minutes of schedule",
       "Number of stop observations by delay in minutes. Synthetic data.", height=520)
save(fig, "04_delay_distribution").show()

## 4. Delays by train

**Ranking metric:** mean arrival delay in minutes over every arrival of the train (origin stops have no arrival and are excluded). Early arrivals count as negative minutes. Ties share a rank.

**Sample size:** `runs` and `arrivals` per train. Trains running six days a week have 39 runs, the others 45.

**Disruptions:** `disrupted_runs` counts runs with any disrupted stop, and `avg_arrival_delay_undisrupted` / `rank_undisrupted` repeat the metric with those runs left out. Section 4.1 explains the difference.

In [5]:
by_train = query("""
    WITH train_stats AS (
        SELECT
            train_id,
            COUNT(DISTINCT service_id) AS runs,
            COUNT(DISTINCT service_id) FILTER (WHERE run_disrupted) AS disrupted_runs,
            COUNT(*) FILTER (WHERE measure = 'arrival') AS arrivals,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'arrival'), 2) AS avg_arrival_delay,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'arrival' AND NOT run_disrupted), 2) AS avg_arrival_delay_undisrupted,
            MEDIAN(delay_min) FILTER (WHERE measure = 'arrival') AS median_arrival_delay,
            MAX(delay_min) FILTER (WHERE measure = 'arrival') AS max_arrival_delay,
            ROUND(100.0 * AVG(is_on_time::INT) FILTER (WHERE measure = 'arrival'), 1) AS arrival_on_time_pct,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'departure'), 2) AS avg_departure_delay,
            ROUND(100.0 * AVG(is_on_time::INT) FILTER (WHERE measure = 'departure'), 1) AS departure_on_time_pct
        FROM delay_obs
        GROUP BY train_id
    )
    SELECT
        -- RANK so ties share a position
        RANK() OVER (ORDER BY avg_arrival_delay DESC) AS delay_rank,
        RANK() OVER (ORDER BY avg_arrival_delay_undisrupted DESC) AS rank_undisrupted,
        train_id,
        runs,
        arrivals,
        disrupted_runs,
        avg_arrival_delay,
        -- against the overall average, not the average of train averages
        ROUND(avg_arrival_delay - (SELECT AVG(delay_min) FROM delay_obs WHERE measure = 'arrival'), 2) AS vs_network_min,
        avg_arrival_delay_undisrupted,
        median_arrival_delay,
        max_arrival_delay,
        arrival_on_time_pct,
        avg_departure_delay,
        departure_on_time_pct
    FROM train_stats
    ORDER BY delay_rank, train_id
""")
by_train

,delay_rank,rank_undisrupted,train_id,runs,arrivals,disrupted_runs,avg_arrival_delay,vs_network_min,avg_arrival_delay_undisrupted,median_arrival_delay,max_arrival_delay,arrival_on_time_pct,avg_departure_delay,departure_on_time_pct
0,1,5,TR007,39,195,4,3.38,0.68,2.06,2.0,27,80.0,4.54,67.7
1,2,2,TR009,45,225,4,3.22,0.52,2.38,3.0,33,75.1,4.31,67.1
2,3,1,TR005,39,234,2,2.93,0.23,2.44,3.0,18,76.1,4.44,63.2
3,4,4,TR006,39,234,3,2.86,0.16,2.17,1.0,24,74.8,3.91,68.8
4,5,7,TR008,45,180,4,2.82,0.12,1.82,2.0,21,80.0,4.56,64.4
5,6,3,TR001,39,156,2,2.81,0.11,2.34,2.0,24,81.4,3.96,67.9
6,7,6,TR010,45,225,3,2.70,0.00,1.94,2.0,30,79.1,4.08,68.0
7,8,9,TR002,45,225,3,2.56,-0.14,1.36,1.0,31,85.3,3.93,70.2
8,9,8,TR004,45,180,1,1.92,-0.78,1.77,1.0,15,80.6,3.26,76.1
9,10,10,TR003,45,180,2,1.53,-1.17,1.17,1.0,16,87.2,3.36,74.4


In [6]:
network_otp = kpi.loc[kpi["measure"] == "arrival", "on_time_pct"].iloc[0]
network_avg = kpi.loc[kpi["measure"] == "arrival", "avg_delay_min"].iloc[0]

# ascending so the worst train ends up on top
ordered = by_train.sort_values("avg_arrival_delay")

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.14,
                    subplot_titles=("Average arrival delay (min)", "On-time arrivals (%)"))
fig.add_trace(go.Bar(
    y=ordered["train_id"], x=ordered["avg_arrival_delay"], orientation="h",
    marker=dict(color=ARRIVAL, cornerradius=4), text=ordered["avg_arrival_delay"].map("{:.2f}".format),
    textposition="outside", textfont=dict(color=INK_SOFT), cliponaxis=False, showlegend=False,
    customdata=ordered[["median_arrival_delay", "max_arrival_delay", "runs", "arrivals", "disrupted_runs",
                        "avg_arrival_delay_undisrupted"]],
    hovertemplate="%{y}<br>Average %{x:.2f} min<br>Median %{customdata[0]:.0f} min<br>"
                  "Maximum %{customdata[1]} min<br>%{customdata[2]} runs, %{customdata[3]} arrivals<br>"
                  "%{customdata[4]} disrupted runs; %{customdata[5]:.2f} min without them<extra></extra>",
), row=1, col=1)
# dots here since the axis doesn't start at 0
fig.add_trace(go.Scatter(
    y=ordered["train_id"], x=ordered["arrival_on_time_pct"], mode="markers+text",
    marker=dict(color=ARRIVAL, size=11, line=dict(color=SURFACE, width=2)),
    text=ordered["arrival_on_time_pct"].map("{:.1f}".format), textposition="middle right",
    textfont=dict(color=INK_SOFT), cliponaxis=False, showlegend=False,
    hovertemplate="%{y}<br>On time %{x:.1f}%<extra></extra>",
), row=1, col=2)
fig.add_vline(x=network_avg, line=dict(color=MUTED, width=1), row=1, col=1)
fig.add_vline(x=network_otp, line=dict(color=MUTED, width=1), row=1, col=2)
fig.add_annotation(x=network_otp, y=1.0, xref="x2", yref="y2 domain", yanchor="bottom", showarrow=False,
                   text=f"Network {network_otp:.1f}%", font=dict(color=INK_SOFT, size=12))
fig.add_annotation(x=network_avg, y=1.0, xref="x", yref="y domain", yanchor="bottom", showarrow=False,
                   text=f"Network {network_avg:.2f}", font=dict(color=INK_SOFT, size=12))
fig.update_xaxes(range=[0, ordered["avg_arrival_delay"].max() * 1.2], row=1, col=1)
fig.update_xaxes(range=[65, 95], row=1, col=2)
fig.update_yaxes(showgrid=False)
fig.update_layout(bargap=0.45)
# push the subplot titles up so they don't overlap the Network labels
fig.update_annotations(selector=dict(yref="paper"), yshift=18, font=dict(color=INK_SOFT, size=13))
titled(fig, "Trains ranked by average arrival delay",
       "Mean arrival delay over all arrivals, most delayed at the top. Hover for sample size and disrupted runs. Synthetic data.",
       height=490).update_layout(margin=dict(t=140))
save(fig, "04_trains_ranked").show()

### 4.1 How disruptions shape the ranking

Arrivals on a disrupted run average about 11 minutes later than on an undisrupted one (13.1 against 1.95 min), so one or two extra disrupted runs can move a train several places. Disruptions fell on the trains roughly at random (notebook 05, section 4), so much of the spread in the ranking reflects where they happened to land:

- TR007 ranks 1st with 4 disrupted runs out of 39; without them it is mid-table.
- TR003 is last on both measures and had 2 disrupted runs.
- With disrupted runs excluded, the trains span about 1.3 minutes, and notebook 05 (section 5) finds that difference is not distinguishable from chance when each run is treated as one observation.

Read the ranking as a description of these 45 days, not as a stable property of the trains.

## 5. Delays by station

In [7]:
by_station = query("""
    -- origin-only stations have no arrivals and drop out. some stations only see one train, small sample
    WITH station_stats AS (
        SELECT
            station_id,
            ANY_VALUE(region) AS region,
            COUNT(DISTINCT train_id) AS trains_serving,
            COUNT(*) FILTER (WHERE measure = 'arrival') AS arrivals,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'arrival'), 2) AS avg_arrival_delay,
            MEDIAN(delay_min) FILTER (WHERE measure = 'arrival') AS median_arrival_delay,
            MAX(delay_min) FILTER (WHERE measure = 'arrival') AS max_arrival_delay,
            ROUND(100.0 * AVG(is_on_time::INT) FILTER (WHERE measure = 'arrival'), 1) AS arrival_on_time_pct,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'departure'), 2) AS avg_departure_delay,
            MAX(delay_min) FILTER (WHERE measure = 'departure') AS max_departure_delay,
            ROUND(100.0 * AVG(is_on_time::INT) FILTER (WHERE measure = 'departure'), 1) AS departure_on_time_pct
        FROM delay_obs
        GROUP BY station_id
    )
    SELECT
        RANK() OVER (ORDER BY avg_arrival_delay DESC) AS delay_rank,
        -- rank within the region too
        RANK() OVER (PARTITION BY region ORDER BY avg_arrival_delay DESC) AS rank_in_region,
        *
    FROM station_stats
    WHERE arrivals > 0
    ORDER BY delay_rank, station_id
""")
by_station

,delay_rank,rank_in_region,station_id,region,trains_serving,arrivals,avg_arrival_delay,median_arrival_delay,max_arrival_delay,arrival_on_time_pct,avg_departure_delay,max_departure_delay,departure_on_time_pct
0,1,1,ST009,North,1,39,3.62,2.0,24,76.9,5.10,28,66.7
1,2,1,ST001,Central,10,90,3.46,2.5,32,75.6,3.57,12,76.2
2,3,2,ST010,North,3,78,3.18,2.0,27,80.8,4.29,11,71.1
3,4,1,ST011,East,3,84,3.12,2.0,31,81.0,3.69,14,68.9
4,5,2,ST019,East,1,39,3.10,3.0,14,69.2,4.26,18,64.1
5,6,2,ST015,Central,3,129,3.01,3.0,33,75.2,4.75,31,59.7
6,7,3,ST007,North,3,123,2.99,2.0,22,74.0,4.63,26,61.8
7,8,1,ST005,West,1,45,2.89,2.0,15,75.6,4.22,17,66.7
8,9,3,ST020,East,1,39,2.85,2.0,16,76.9,4.74,20,59.0
9,10,4,ST012,East,3,129,2.81,2.0,30,79.8,4.64,29,63.6


In [8]:
ordered = by_station.sort_values("avg_arrival_delay")

fig = go.Figure(go.Bar(
    y=ordered["station_id"], x=ordered["avg_arrival_delay"], orientation="h",
    marker=dict(color=ARRIVAL, cornerradius=4), text=ordered["avg_arrival_delay"].map("{:.2f}".format),
    textposition="outside", textfont=dict(color=INK_SOFT), cliponaxis=False,
    customdata=ordered[["region", "arrivals", "arrival_on_time_pct", "max_arrival_delay"]],
    hovertemplate="%{y} (%{customdata[0]})<br>Average %{x:.2f} min<br>On time %{customdata[2]:.1f}%<br>"
                  "Maximum %{customdata[3]} min<br>%{customdata[1]} arrivals<extra></extra>",
))
fig.add_vline(x=network_avg, line=dict(color=MUTED, width=1))
fig.add_annotation(x=network_avg, y=1.0, yref="y domain", yanchor="bottom", showarrow=False,
                   text=f"Network {network_avg:.2f}", font=dict(color=INK_SOFT, size=12))
fig.update_xaxes(title_text="Average arrival delay (minutes)", range=[0, ordered["avg_arrival_delay"].max() * 1.15])
fig.update_yaxes(showgrid=False)
fig.update_layout(bargap=0.45)
titled(fig, "Stations ranked by average arrival delay",
       "Most delayed station at the top. Hover for sample size. Synthetic data.", height=640).show()

## 6. Delays by date

In [9]:
by_date = query("""
    WITH daily AS (
        SELECT
            service_date,
            COUNT(*) AS arrivals,
            SUM(delay_min) AS total_delay_min,
            SUM(is_on_time::INT) AS on_time_arrivals,
            COUNT(*) FILTER (WHERE is_disrupted) AS disrupted_arrivals
        FROM delay_obs
        WHERE measure = 'arrival'
        GROUP BY service_date
    )
    SELECT
        service_date,
        dayname(service_date) AS weekday,
        arrivals,
        ROUND(total_delay_min / arrivals, 2) AS avg_arrival_delay,
        ROUND(100.0 * on_time_arrivals / arrivals, 1) AS on_time_pct,
        disrupted_arrivals,
        -- rolling sum / rolling sum so each day is weighted by its arrivals
        -- null until there's a full 7 days
        CASE WHEN COUNT(*) OVER week = 7
            THEN ROUND(SUM(total_delay_min) OVER week / SUM(arrivals) OVER week, 2) END AS avg_delay_7d,
        CASE WHEN COUNT(*) OVER week = 7
            THEN ROUND(100.0 * SUM(on_time_arrivals) OVER week / SUM(arrivals) OVER week, 1) END AS on_time_pct_7d
    FROM daily
    WINDOW week AS (ORDER BY service_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
    ORDER BY service_date
""")
by_date.head(10)

,service_date,weekday,arrivals,avg_arrival_delay,on_time_pct,disrupted_arrivals,avg_delay_7d,on_time_pct_7d
0,2026-08-01,Saturday,48,6.04,50.0,10,NaN,NaN
1,2026-08-02,Sunday,48,2.94,75.0,0,NaN,NaN
2,2026-08-03,Monday,48,2.27,83.3,0,NaN,NaN
3,2026-08-04,Tuesday,37,1.43,89.2,0,NaN,NaN
4,2026-08-05,Wednesday,48,1.94,81.3,4,NaN,NaN
5,2026-08-06,Thursday,48,2.90,72.9,0,NaN,NaN
6,2026-08-07,Friday,38,5.66,65.8,10,3.30,73.7
7,2026-08-08,Saturday,48,1.71,83.3,0,2.64,78.7
8,2026-08-09,Sunday,48,1.15,91.7,0,2.37,81.3
9,2026-08-10,Monday,48,2.23,81.3,4,2.36,81.0


In [10]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Average arrival delay (min)", "On-time arrivals (%)"))
for row, (daily_col, rolling_col, unit) in enumerate(
        [("avg_arrival_delay", "avg_delay_7d", " min"), ("on_time_pct", "on_time_pct_7d", "%")], start=1):
    # daily line in grey, rolling average on top
    fig.add_trace(go.Scatter(
        x=by_date["service_date"], y=by_date[daily_col], name="Daily", mode="lines",
        line=dict(color=AXIS, width=1.5), legendgroup="daily", showlegend=row == 1,
        hovertemplate="%{x|%a %d %b}: %{y}" + unit + "<extra>Daily</extra>",
    ), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=by_date["service_date"], y=by_date[rolling_col], name="7-day average", mode="lines",
        line=dict(color=ARRIVAL, width=2), legendgroup="rolling", showlegend=row == 1,
        hovertemplate="%{x|%a %d %b}: %{y}" + unit + "<extra>7-day average</extra>",
    ), row=row, col=1)
fig.update_xaxes(tickformat="%d %b")
fig.update_layout(hovermode="x unified")
fig.update_annotations(font=dict(color=INK_SOFT, size=13), x=0, xanchor="left")
titled(fig, "Daily arrival delay and on-time performance",
       "Arrival delay and on-time performance by service date. Synthetic data.",
       height=580).update_layout(legend=dict(y=1.05), margin=dict(t=120))
save(fig, "04_daily_trend").show()

In [11]:
# 5 worst and 5 best days
query("""
    WITH daily AS (
        SELECT
            service_date,
            dayname(service_date) AS weekday,
            COUNT(*) AS arrivals,
            ROUND(AVG(delay_min), 2) AS avg_arrival_delay,
            MAX(delay_min) AS max_arrival_delay,
            ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct,
            COUNT(*) FILTER (WHERE is_disrupted) AS disrupted_arrivals
        FROM delay_obs
        WHERE measure = 'arrival'
        GROUP BY service_date
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (ORDER BY avg_arrival_delay DESC, service_date) AS worst_rank,
            ROW_NUMBER() OVER (ORDER BY avg_arrival_delay, service_date) AS best_rank
        FROM daily
    )
    SELECT
        CASE WHEN worst_rank <= 5 THEN 'worst' ELSE 'best' END AS day_group,
        LEAST(worst_rank, best_rank) AS rank_in_group,
        service_date, weekday, arrivals, avg_arrival_delay, max_arrival_delay, on_time_pct, disrupted_arrivals
    FROM ranked
    WHERE worst_rank <= 5 OR best_rank <= 5
    ORDER BY day_group DESC, rank_in_group
""")

,day_group,rank_in_group,service_date,weekday,arrivals,avg_arrival_delay,max_arrival_delay,on_time_pct,disrupted_arrivals
0,worst,1,2026-08-01,Saturday,48,6.04,24,50.0,10
1,worst,2,2026-08-07,Friday,38,5.66,24,65.8,10
2,worst,3,2026-09-01,Tuesday,37,5.65,30,67.6,11
3,worst,4,2026-08-28,Friday,38,5.03,19,52.6,10
4,worst,5,2026-08-13,Thursday,48,5.02,30,64.6,5
5,best,1,2026-09-09,Wednesday,48,0.67,8,89.6,0
6,best,2,2026-09-02,Wednesday,48,1.00,9,93.8,0
7,best,3,2026-08-18,Tuesday,37,1.05,12,97.3,0
8,best,4,2026-08-09,Sunday,48,1.15,7,91.7,0
9,best,5,2026-09-07,Monday,48,1.25,16,87.5,5


## 7. Patterns by day of week and time of day

### 7.1 Day of week

In [12]:
by_weekday = query("""
    -- only 6-7 dates per weekday and the trains differ by day, so don't read too much into it
    SELECT
        isodow(service_date) AS weekday_no,
        dayname(service_date) AS weekday,
        COUNT(DISTINCT service_date) AS dates_observed,
        COUNT(DISTINCT train_id) AS trains_running,
        COUNT(*) AS arrivals,
        ROUND(AVG(delay_min), 2) AS avg_arrival_delay,
        MEDIAN(delay_min) AS median_arrival_delay,
        MAX(delay_min) AS max_arrival_delay,
        ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct,
        ROUND(100.0 * AVG(is_disrupted::INT), 1) AS disrupted_pct
    FROM delay_obs
    WHERE measure = 'arrival'
    GROUP BY ALL
    ORDER BY weekday_no
""")
by_weekday

,weekday_no,weekday,dates_observed,trains_running,arrivals,avg_arrival_delay,median_arrival_delay,max_arrival_delay,on_time_pct,disrupted_pct
0,1,Monday,7,10,336,2.68,2.0,33,79.8,8.6
1,2,Tuesday,6,8,222,2.61,1.0,30,83.8,6.8
2,3,Wednesday,6,10,288,1.84,1.5,18,84.7,3.1
3,4,Thursday,6,10,288,3.30,2.0,31,74.3,7.6
4,5,Friday,6,8,228,3.72,3.0,24,71.5,8.8
5,6,Saturday,7,10,336,2.61,2.0,27,81.8,6.0
6,7,Sunday,7,10,336,2.39,2.0,19,80.7,6.3


In [13]:
fig = go.Figure(go.Bar(
    x=by_weekday["weekday"], y=by_weekday["avg_arrival_delay"], width=0.22,
    marker=dict(color=ARRIVAL, cornerradius=4), text=by_weekday["avg_arrival_delay"].map("{:.2f}".format),
    textposition="outside", textfont=dict(color=INK_SOFT), cliponaxis=False,
    customdata=by_weekday[["on_time_pct", "arrivals", "dates_observed", "disrupted_pct"]],
    hovertemplate="%{x}<br>Average %{y:.2f} min<br>On time %{customdata[0]:.1f}%<br>"
                  "Disrupted %{customdata[3]:.1f}%<br>%{customdata[1]} arrivals on %{customdata[2]} dates<extra></extra>",
))
fig.add_hline(y=network_avg, line=dict(color=MUTED, width=1))
fig.add_annotation(x=1, xref="x domain", xanchor="right", y=network_avg, yanchor="bottom", showarrow=False,
                   text=f"Network {network_avg:.2f}", font=dict(color=INK_SOFT, size=12))
fig.update_yaxes(title_text="Average arrival delay (minutes)", range=[0, by_weekday["avg_arrival_delay"].max() * 1.2])
fig.update_xaxes(showgrid=False)
titled(fig, "Average arrival delay by day of week",
       "Six or seven dates per weekday, so differences are indicative only. Synthetic data.").show()

### 7.2 Time of day

In [14]:
# 6h bands instead of hours: each train runs at a fixed time, so hourly would just compare trains
TIME_BAND_SQL = """
    CASE
        WHEN hour(scheduled_time) < 6 THEN 'Night 00-06'
        WHEN hour(scheduled_time) < 12 THEN 'Morning 06-12'
        WHEN hour(scheduled_time) < 18 THEN 'Afternoon 12-18'
        ELSE 'Evening 18-24'
    END
"""
BAND_ORDER = ["Night 00-06", "Morning 06-12", "Afternoon 12-18", "Evening 18-24"]

by_band = query(f"""
    SELECT
        {TIME_BAND_SQL} AS time_band,
        COUNT(DISTINCT train_id) AS trains,
        COUNT(*) AS arrivals,
        ROUND(AVG(delay_min), 2) AS avg_arrival_delay,
        MEDIAN(delay_min) AS median_arrival_delay,
        MAX(delay_min) AS max_arrival_delay,
        ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct
    FROM delay_obs
    WHERE measure = 'arrival'
    GROUP BY time_band
    -- clock order, not alphabetical
    ORDER BY MIN(hour(scheduled_time))
""")
by_band

,time_band,trains,arrivals,avg_arrival_delay,median_arrival_delay,max_arrival_delay,on_time_pct
0,Night 00-06,4,516,3.00,1.5,33,77.5
1,Morning 06-12,4,559,2.47,2.0,24,81.8
2,Afternoon 12-18,3,251,2.86,2.0,27,80.1
3,Evening 18-24,6,708,2.61,2.0,30,79.5


In [15]:
weekday_band = query(f"""
    SELECT
        isodow(service_date) AS weekday_no,
        dayname(service_date) AS weekday,
        {TIME_BAND_SQL} AS time_band,
        COUNT(*) AS arrivals,
        ROUND(AVG(delay_min), 2) AS avg_arrival_delay
    FROM delay_obs
    WHERE measure = 'arrival'
    GROUP BY ALL
    ORDER BY weekday_no
""")

# pivot to weekday x band for the heatmap
weekdays = by_weekday["weekday"].tolist()
grid = weekday_band.pivot(index="weekday", columns="time_band", values="avg_arrival_delay").loc[weekdays, BAND_ORDER]
counts = weekday_band.pivot(index="weekday", columns="time_band", values="arrivals").loc[weekdays, BAND_ORDER]

# blank out cells with too few arrivals
MIN_CELL_ARRIVALS = 30
shown = grid.where(counts >= MIN_CELL_ARRIVALS)
labels = shown.map(lambda value: "" if pd.isna(value) else f"{value:.1f}")

fig = go.Figure(go.Heatmap(
    z=shown.values, x=BAND_ORDER, y=weekdays, customdata=counts.values, text=labels.values,
    colorscale=[[i / (len(BLUE_RAMP) - 1), c] for i, c in enumerate(BLUE_RAMP)],
    xgap=2, ygap=2, texttemplate="%{text}", hoverongaps=False,
    colorbar=dict(title=dict(text="Avg delay<br>(min)", font=dict(color=MUTED, size=12)),
                  thickness=12, outlinewidth=0, tickfont=dict(color=MUTED)),
    hovertemplate="%{y}, %{x}<br>Average %{z:.2f} min<br>%{customdata} arrivals<extra></extra>",
))
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(showgrid=False, side="bottom")
titled(fig, "Average arrival delay by weekday and time of day",
       f"Minutes. Cells with fewer than {MIN_CELL_ARRIVALS} arrivals are left blank. Synthetic data.",
       height=440)
save(fig, "04_weekday_time_heatmap").show()

## 8. Where delay builds up

### 8.1 Running against standing

In [16]:
query("""
    WITH steps AS (
        SELECT
            service_id,
            station_sequence,
            -- arrival delay minus previous departure delay = change while running
            arrival_delay_min - LAG(departure_delay_min) OVER run AS change_running,
            -- departure minus arrival at the same stop = change while standing
            departure_delay_min - arrival_delay_min AS change_standing
        FROM train_run_events_clean
        WINDOW run AS (PARTITION BY service_id ORDER BY station_sequence)
    ),
    long AS (
        SELECT 'Running between stops' AS phase, change_running AS change_min FROM steps
        UNION ALL
        SELECT 'Standing at a stop', change_standing FROM steps
    )
    SELECT
        phase,
        COUNT(change_min) AS observations,
        ROUND(AVG(change_min), 2) AS avg_change_min,
        MEDIAN(change_min) AS median_change_min,
        ROUND(100.0 * AVG((change_min > 0)::INT), 1) AS pct_delay_grew,
        ROUND(100.0 * AVG((change_min < 0)::INT), 1) AS pct_delay_shrank
    FROM long
    WHERE change_min IS NOT NULL
    GROUP BY phase
    ORDER BY phase
""")

,phase,observations,avg_change_min,median_change_min,pct_delay_grew,pct_delay_shrank
0,Running between stops,2034,-1.35,-1.0,32.1,58.6
1,Standing at a stop,1608,1.51,1.5,63.6,22.9


### 8.2 Delay along a run

In [17]:
along_run = query("""
    -- first 5 stops only, every run has at least that many
    WITH events AS (
        -- arrive at stop k = 2k - 1, depart from stop k = 2k
        SELECT measure, station_sequence, delay_min,
            station_sequence * 2 - (measure = 'arrival')::INT AS event_order
        FROM delay_obs
        WHERE (measure = 'departure' AND station_sequence <= 4)
           OR (measure = 'arrival' AND station_sequence BETWEEN 2 AND 5)
    ),
    profile AS (
        SELECT event_order, measure, station_sequence, COUNT(*) AS observations, AVG(delay_min) AS avg_delay
        FROM events
        GROUP BY event_order, measure, station_sequence
    )
    SELECT
        event_order,
        CASE WHEN measure = 'arrival' THEN 'Arrive ' ELSE 'Depart ' END || station_sequence AS event,
        measure,
        observations,
        ROUND(avg_delay, 2) AS avg_delay_min,
        ROUND(avg_delay - LAG(avg_delay) OVER (ORDER BY event_order), 2) AS change_from_previous
    FROM profile
    ORDER BY event_order
""")
along_run

,event_order,event,measure,observations,avg_delay_min,change_from_previous
0,2,Depart 1,departure,426,3.66,NaN
1,3,Arrive 2,arrival,426,2.31,-1.35
2,4,Depart 2,departure,426,3.84,1.53
3,5,Arrive 3,arrival,426,2.54,-1.30
4,6,Depart 3,departure,426,4.14,1.60
5,7,Arrive 4,arrival,426,2.77,-1.37
6,8,Depart 4,departure,426,4.29,1.52
7,9,Arrive 5,arrival,426,2.95,-1.34


In [18]:
fig = go.Figure()
# grey line joining the points, markers coloured by measure
fig.add_trace(go.Scatter(x=along_run["event"], y=along_run["avg_delay_min"], mode="lines",
                         line=dict(color=AXIS, width=2), showlegend=False, hoverinfo="skip"))
for measure, label, colour in [("arrival", "Arrival", ARRIVAL), ("departure", "Departure", DEPARTURE)]:
    part = along_run[along_run["measure"] == measure]
    fig.add_trace(go.Scatter(
        x=part["event"], y=part["avg_delay_min"], name=label, mode="markers+text",
        marker=dict(color=colour, size=12, line=dict(color=SURFACE, width=2)),
        text=part["avg_delay_min"].map("{:.2f}".format),
        textposition="bottom center" if measure == "arrival" else "top center", textfont=dict(color=INK_SOFT),
        hovertemplate="%{x}<br>Average delay %{y:.2f} min<extra></extra>",
    ))
fig.update_xaxes(categoryorder="array", categoryarray=along_run["event"].tolist(), showgrid=False)
fig.update_yaxes(title_text="Average delay (minutes)", range=[0, along_run["avg_delay_min"].max() * 1.25])
titled(fig, "Delay is added at stops and recovered between them",
       "Average delay at each event over the first five stops of a run. Synthetic data.").show()

## 9. Disruptions

In [19]:
by_disruption = query("""
    SELECT
        disruption_type,
        COUNT(*) AS arrivals,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_arrivals,
        ROUND(AVG(delay_min), 2) AS avg_arrival_delay,
        MEDIAN(delay_min) AS median_arrival_delay,
        MAX(delay_min) AS max_arrival_delay,
        ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct,
        COUNT(*) FILTER (WHERE NOT is_on_time) AS late_arrivals,
        -- share of all late arrivals
        ROUND(100.0 * COUNT(*) FILTER (WHERE NOT is_on_time)
            / SUM(COUNT(*) FILTER (WHERE NOT is_on_time)) OVER (), 1) AS pct_of_late_arrivals
    FROM delay_obs
    WHERE measure = 'arrival'
    GROUP BY disruption_type
    ORDER BY avg_arrival_delay DESC, disruption_type
""")
by_disruption

,disruption_type,arrivals,pct_of_arrivals,avg_arrival_delay,median_arrival_delay,max_arrival_delay,on_time_pct,late_arrivals,pct_of_late_arrivals
0,track,30,1.5,15.40,14.5,33,16.7,25,6.1
1,operational,28,1.4,12.86,12.5,31,25.0,21,5.1
2,signal,43,2.1,12.56,11.0,30,16.3,36,8.7
3,weather,35,1.7,12.03,12.0,27,8.6,32,7.7
4,none,1898,93.3,1.95,1.0,15,84.2,299,72.4


In [20]:
ordered = by_disruption.sort_values("avg_arrival_delay")

fig = go.Figure(go.Bar(
    y=ordered["disruption_type"].str.capitalize(), x=ordered["avg_arrival_delay"], orientation="h",
    marker=dict(color=ARRIVAL, cornerradius=4), text=ordered["avg_arrival_delay"].map("{:.2f}".format),
    textposition="outside", textfont=dict(color=INK_SOFT), cliponaxis=False,
    customdata=ordered[["arrivals", "on_time_pct", "pct_of_late_arrivals"]],
    hovertemplate="%{y}<br>Average %{x:.2f} min<br>On time %{customdata[1]:.1f}%<br>"
                  "%{customdata[0]} arrivals<br>%{customdata[2]:.1f}% of all late arrivals<extra></extra>",
))
fig.update_xaxes(title_text="Average arrival delay (minutes)", range=[0, ordered["avg_arrival_delay"].max() * 1.15])
fig.update_yaxes(showgrid=False)
fig.update_layout(bargap=0.5)
titled(fig, "Disrupted stops arrive far later than undisrupted ones",
       "Average arrival delay by simulated disruption type. Synthetic data.", height=360).show()

## 10. Most delayed runs

In [21]:
query("""
    -- one row per run
    WITH run_stats AS (
        SELECT
            service_id,
            train_id,
            service_date,
            MAX(delay_min) FILTER (WHERE measure = 'arrival') AS max_arrival_delay,
            ROUND(AVG(delay_min) FILTER (WHERE measure = 'arrival'), 1) AS avg_arrival_delay,
            -- delay at the last stop
            arg_max(delay_min, station_sequence) FILTER (WHERE measure = 'arrival') AS final_arrival_delay,
            COUNT(DISTINCT station_id) FILTER (WHERE is_disrupted) AS disrupted_stops,
            COALESCE(string_agg(DISTINCT disruption_type, ', ') FILTER (WHERE is_disrupted), 'none') AS disruption_types
        FROM delay_obs
        GROUP BY service_id, train_id, service_date
    )
    SELECT
        -- service_id as the tiebreaker
        ROW_NUMBER() OVER (ORDER BY final_arrival_delay DESC, max_arrival_delay DESC, service_id) AS run_rank,
        *
    FROM run_stats
    QUALIFY run_rank <= 10
    ORDER BY run_rank
""")

,run_rank,service_id,train_id,service_date,max_arrival_delay,avg_arrival_delay,final_arrival_delay,disrupted_stops,disruption_types
0,1,RUN000367,TR009,2026-08-31,33,25.8,32,5,track
1,2,RUN000073,TR002,2026-09-03,31,24.8,31,5,operational
2,3,RUN000052,TR002,2026-08-13,30,21.0,30,5,signal
3,4,RUN000413,TR010,2026-09-01,30,18.0,30,5,track
4,5,RUN000277,TR007,2026-08-29,27,18.6,27,5,weather
5,6,RUN000001,TR001,2026-08-01,24,15.3,24,4,weather
6,7,RUN000241,TR006,2026-09-01,24,17.3,22,6,track
7,8,RUN000258,TR007,2026-08-07,24,19.0,21,5,signal
8,9,RUN000302,TR008,2026-08-11,21,17.0,21,4,track
9,10,RUN000308,TR008,2026-08-17,20,14.3,20,4,operational


## 11. Key findings

Synthetic data: illustrative, not evidence about any real service.

| # | Finding | Evidence |
| --- | --- | --- |
| 1 | Arrivals are usually close to schedule | Avg 2.70 min, median 2, max 33; 79.7% on time |
| 2 | Departures run later than arrivals | Avg 4.05 min, median 4, max 36; 68.6% on time |
| 3 | Delay is added at stops and recovered on the move | +1.51 min standing, -1.35 min running |
| 4 | Highest mean arrival delay: `TR007`, `TR009` | 3.38 and 3.22 min over 195 and 225 arrivals; `TR003` lowest at 1.53. Sensitive to disruptions: `TR007` is mid-table without its 4 disrupted runs (section 4.1) |
| 5 | Most delayed stations: `ST009`, `ST001`, `ST010` | 3.62, 3.46 and 3.18 min; `ST009` is a small sample |
| 6 | Bad days are disruption days | Daily avg 0.67 to 6.04 min; the five worst days have 5 to 11 disrupted arrivals each; the five best have 0 to 5 |
| 7 | Disruptions are rare but severe | 6.7% of arrivals, 13.11 min late against 1.95 |
| 8 | Most late arrivals are still ordinary ones | 72.4% of late arrivals carry no disruption flag |
| 9 | Friday is weakest, Wednesday strongest | 3.72 min / 71.5% against 1.84 min / 84.7%; only 6 dates each, not significant at the date level (notebook 05) |
| 10 | No clear time-of-day pattern | Bands range from 2.47 to 3.00 min |

**Limits:** 45 days, 10 trains. This notebook is descriptive; significance tests are in notebook 05, which finds the train and weekday differences are not reliable. Weekday and time-of-day gaps partly reflect which trains run. The 5 minute threshold is an assumption.

In [22]:
con.close()
print("Connection closed.")

Connection closed.
